# P5 — Elexon Generation Data: Full Extraction & Spatial Split

Locked decisions this notebook builds against (not re-litigated here):

- **Sources**: Elexon Insights Solution API only (public, no key, Stream endpoints) — never the legacy key-gated BMRS API or ElexonDataPortal package.
- **Fuel-type scope**: mixed GB fleet, all fuel types. This is a weaker geographic story than a single fuel type would give — that tension is acknowledged here and must be stated explicitly in the eventual report, not glossed over.
- **Spatial unit**: site (`dictionary_id`), not individual BMU. Multi-BMU sites (e.g. Didcot) are aggregated to one spatial point, not treated as separate points.
- **Resolution/span**: half-hourly, full available history. The exploration notebook found an apparent ~2019-02-01 backfill floor on a 6-site sample — this notebook confirms or corrects that at full-fleet scale rather than assuming it generalises.
- **Raw storage**: one CSV per BMU under `data/raw/generation/`, source of truth. A consolidated parquet file is a later, disposable, regenerable convenience cache for the training notebook — never the source of truth.
- **Empty/mothballed BMUs**: detected and logged generically via the manifest (zero rows returned), not hand-excluded by name.

This notebook stops short of: finalising the fuel-type stratification threshold, applying a minimum-history cutoff, or treating the parquet cache as anything other than derived. Those are for Simon to confirm once the coverage and history-length reports below are available.

In [1]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO
from datetime import date, datetime, timedelta
import time
import json

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RAW_DIR = Path("../data/raw/generation")
RAW_DIR.mkdir(parents=True, exist_ok=True)
REF_DIR = Path("../data/reference")
REF_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

OSUKED_RAW = "https://raw.githubusercontent.com/OSUKED/Power-Station-Dictionary/main"
ELEXON_API = "https://data.elexon.co.uk/bmrs/api/v1"

REQUEST_DELAY_S = 0.1  # be a good netizen
MANIFEST_PATH = INTERIM_DIR / "extraction_manifest.csv"

# Global chunk floor: deliberately earlier than the ~2019-02-01 floor the
# 6-site exploration sample suggested, so the full pull can confirm or
# correct that rather than assume it. See Step 6 for the confirmed figure.
EXTRACTION_START_YEAR = 2015
EXTRACTION_END = date.today() - timedelta(days=10)  # ~5 working day publication lag

## Step 1 — Resumable extraction pipeline design

**Manifest** (`data/interim/extraction_manifest.csv`): one row per `(bmu, chunk_start, chunk_end)` attempted. Columns: `ngc_bmu_id`, `chunk_start`, `chunk_end`, `status` (`pending`/`success`/`failed`), `rows_returned`, `attempted_at`, `error`.

**Chunking**: one calendar year per chunk, per BMU, from `EXTRACTION_START_YEAR` to the present (minus the publication lag). This keeps individual requests small and makes retries cheap.

**Resume logic**: the manifest is loaded (not rebuilt) on every run. Rows already `success` are skipped. Rows `failed` or `pending` are retried/attempted. This means an interrupted run can simply be re-run.

**Raw storage**: each successful chunk's rows are appended straight to `data/raw/generation/{ngc_bmu_id}.csv` as soon as they're fetched — so data already pulled is durable on disk even if the run is interrupted mid-way, not just held in memory until the end.

**Retries**: HTTP 429/5xx are treated as transient — retried with a short backoff (up to 3 attempts) rather than marked `failed` immediately. Other errors (4xx other than 429) are logged as `failed` and not retried automatically.

**Politeness**: a fixed 0.1s delay between requests.

In [2]:
# Fresh pull of the OSUKED reference tables (the complete BMU universe for this
# extraction comes from fuel_types, per the brief - not just the exploration sample)
osuked_files = {
    "fuel_types": "data/linked-datapackages/bmu-fuel-types/fuel_types.csv",
    "plant_locations": "data/linked-datapackages/plant-locations/plant-locations.csv",
    "dictionary_ids": "data/dictionary/ids.csv",
}

osuked = {}
for name, path in osuked_files.items():
    resp = requests.get(f"{OSUKED_RAW}/{path}", timeout=30)
    resp.raise_for_status()
    osuked[name] = pd.read_csv(StringIO(resp.text))

bmu_universe = sorted(osuked["fuel_types"]["ngc_bmu_id"].dropna().str.strip().unique())
print(f"Full BMU universe from OSUKED fuel_types: {len(bmu_universe)} distinct BMUs")

Full BMU universe from OSUKED fuel_types: 462 distinct BMUs


In [3]:
MANIFEST_COLUMNS = ["ngc_bmu_id", "chunk_start", "chunk_end", "status", "rows_returned", "attempted_at", "error"]


def year_chunks(start_year: int, end_date: date):
    """Yield (chunk_start, chunk_end) date strings, one per calendar year."""
    chunks = []
    for year in range(start_year, end_date.year + 1):
        chunk_start = date(year, 1, 1)
        chunk_end = date(year, 12, 31) if year < end_date.year else end_date
        if chunk_start > end_date:
            break
        chunks.append((chunk_start.isoformat(), chunk_end.isoformat()))
    return chunks


def load_or_init_manifest(bmus, start_year=EXTRACTION_START_YEAR, end_date=EXTRACTION_END):
    if MANIFEST_PATH.exists():
        manifest = pd.read_csv(MANIFEST_PATH, dtype={"ngc_bmu_id": str})
    else:
        manifest = pd.DataFrame(columns=MANIFEST_COLUMNS)

    existing_keys = set(zip(manifest["ngc_bmu_id"], manifest["chunk_start"], manifest["chunk_end"]))
    new_rows = []
    for bmu in bmus:
        for chunk_start, chunk_end in year_chunks(start_year, end_date):
            key = (bmu, chunk_start, chunk_end)
            if key not in existing_keys:
                new_rows.append({
                    "ngc_bmu_id": bmu, "chunk_start": chunk_start, "chunk_end": chunk_end,
                    "status": "pending", "rows_returned": pd.NA, "attempted_at": pd.NA, "error": pd.NA,
                })
    if new_rows:
        manifest = pd.concat([manifest, pd.DataFrame(new_rows)], ignore_index=True)
        manifest.to_csv(MANIFEST_PATH, index=False)
    return manifest


def save_manifest(manifest):
    manifest.to_csv(MANIFEST_PATH, index=False)

In [4]:
RETRYABLE_STATUS = {429, 500, 502, 503, 504}


def fetch_chunk(bmu, chunk_start, chunk_end, max_attempts=3):
    """Fetch one (bmu, year) chunk from the B1610 stream endpoint.
    Retries 429/5xx with backoff; other HTTP errors (e.g. 400/404) fail immediately.
    """
    params = {"from": chunk_start, "to": chunk_end, "bmUnit": bmu}
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            r = requests.get(f"{ELEXON_API}/datasets/B1610/stream", params=params, timeout=60)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            last_exc = exc
            if attempt < max_attempts:
                time.sleep(2 ** attempt)  # 2s, 4s backoff
            continue

        if r.status_code in RETRYABLE_STATUS:
            last_exc = requests.exceptions.HTTPError(f"retryable status {r.status_code}")
            if attempt < max_attempts:
                time.sleep(2 ** attempt)
            continue

        r.raise_for_status()  # non-retryable 4xx raises immediately, no retry loop
        return r.json()
    raise last_exc


def append_bmu_csv(bmu, records):
    if not records:
        return
    df = pd.DataFrame(records)
    path = RAW_DIR / f"{bmu}.csv"
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)

In [5]:
def run_extraction(manifest, progress_every=25):
    """Work through every pending/failed row in the manifest, in place.
    Prints a progress line with a rough ETA every `progress_every` chunks attempted.
    Safe to interrupt and re-run: already-`success` rows are untouched here.
    """
    todo_mask = manifest["status"].isin(["pending", "failed"])
    todo_idx = manifest.index[todo_mask].tolist()
    total_todo = len(todo_idx)
    start_time = time.monotonic()
    total_rows_this_run = 0

    print(f"{total_todo} chunks to attempt ({(~todo_mask).sum()} already succeeded previously)")

    for n, idx in enumerate(todo_idx, start=1):
        row = manifest.loc[idx]
        bmu, chunk_start, chunk_end = row["ngc_bmu_id"], row["chunk_start"], row["chunk_end"]
        try:
            records = fetch_chunk(bmu, chunk_start, chunk_end)
            append_bmu_csv(bmu, records)
            manifest.loc[idx, ["status", "rows_returned", "attempted_at", "error"]] = (
                "success", len(records), datetime.now().isoformat(timespec="seconds"), pd.NA,
            )
            total_rows_this_run += len(records)
        except Exception as exc:
            manifest.loc[idx, ["status", "rows_returned", "attempted_at", "error"]] = (
                "failed", pd.NA, datetime.now().isoformat(timespec="seconds"), str(exc)[:200],
            )
        time.sleep(REQUEST_DELAY_S)

        if n % progress_every == 0 or n == total_todo:
            save_manifest(manifest)  # flush periodically so progress is durable, not just in memory
            elapsed = time.monotonic() - start_time
            rate = n / elapsed  # chunks/sec
            remaining = total_todo - n
            eta_s = remaining / rate if rate > 0 else float("nan")
            eta_min = eta_s / 60
            done_bmus = manifest.loc[manifest["status"] == "success", "ngc_bmu_id"].nunique()
            print(
                f"[{n}/{total_todo}] {n/total_todo:.1%} | "
                f"{done_bmus}/{len(bmu_universe)} BMUs with >=1 success | "
                f"{total_rows_this_run:,} rows this run | "
                f"{rate:.2f} chunks/s | ETA ~{eta_min:.1f} min"
            )

    save_manifest(manifest)
    return manifest

## Step 2 — Kick off the full pull

Across the complete BMU universe (all `ngc_bmu_id` values in OSUKED `fuel_types`), not just the 6-BMU exploration sample. This is expected to run long (hundreds of BMUs × up to 12 yearly chunks each) — that's fine, the manifest makes it resumable, and the progress line every 25 chunks includes a rate-based ETA so it's easy to tell it's making steady progress rather than stuck.

In [6]:
manifest = load_or_init_manifest(bmu_universe)
print(f"Manifest: {len(manifest)} total chunks, {(manifest['status'] == 'success').sum()} already successful")

manifest = run_extraction(manifest)

print()
print("Pull finished (or manifest fully attempted this run).")
print(manifest["status"].value_counts())

Manifest: 5544 total chunks, 5544 already successful
0 chunks to attempt (5544 already succeeded previously)

Pull finished (or manifest fully attempted this run).
status
success    5544
Name: count, dtype: int64


## Step 3 — Site-level aggregation

Rebuilding the confirmed join path from the exploration notebook: `plant_locations.dictionary_id -> dictionary_ids.dictionary_id -> dictionary_ids.ngc_bmu_id (exploded) -> fuel_types.ngc_bmu_id`. This gives every BMU's owning site (`dictionary_id`) and coordinates.

**Multi-BMU sites are combined by summing generation output** (`quantity`, in MWh) across all BMUs belonging to the same `dictionary_id`, matched on `settlementDate` + `settlementPeriod`. This treats a site like Didcot as a single generator equal to the sum of its units — the natural reading of "site-level generation," and consistent with the spatial-unit-is-site decision.

This assumption is questionable if a site's BMUs span genuinely different fuel types (summing a wind turbine and a gas peaker under one coordinate would blur the fuel-type signal that's the whole point of this dataset) — checked explicitly below, not assumed away.

In [7]:
# Rebuild the join path from exploration
ids = osuked["dictionary_ids"][["dictionary_id", "name", "ngc_bmu_id"]].copy()
ids = ids.dropna(subset=["ngc_bmu_id"])
ids["ngc_bmu_id"] = ids["ngc_bmu_id"].str.split(",")
ids_exploded = ids.explode("ngc_bmu_id")
ids_exploded["ngc_bmu_id"] = ids_exploded["ngc_bmu_id"].str.strip()

site_bmu_map = (
    osuked["plant_locations"]
    .merge(ids_exploded, on="dictionary_id", how="inner")
    .merge(osuked["fuel_types"], on="ngc_bmu_id", how="inner")
)
# Only keep BMUs we actually attempted to pull data for
site_bmu_map = site_bmu_map[site_bmu_map["ngc_bmu_id"].isin(bmu_universe)]

print(f"site_bmu_map: {len(site_bmu_map)} BMU rows across {site_bmu_map['dictionary_id'].nunique()} sites")
site_bmu_map.head()

site_bmu_map: 403 BMU rows across 213 sites


,dictionary_id,longitude,latitude,name,ngc_bmu_id,fuel_type,comments
0,10000,-3.603516,57.480403,Rothes Bio-Plant CHP,MARK-1,BIOMASS,NaN
1,10000,-3.603516,57.480403,Rothes Bio-Plant CHP,MARK-2,BIOMASS,NaN
2,10001,-1.267570,51.623630,Didcot,DIDC01G,OCGT,NaN
3,10001,-1.267570,51.623630,Didcot,DIDC02G,OCGT,NaN
4,10001,-1.267570,51.623630,Didcot,DIDC03G,OCGT,NaN


In [8]:
# Flag sites whose BMUs span more than one fuel type - the sum-assumption is
# questionable there, since it would blend distinct generation signals
fuel_types_per_site = site_bmu_map.groupby("dictionary_id")["fuel_type"].nunique()
mixed_fuel_sites = fuel_types_per_site[fuel_types_per_site > 1].index

if len(mixed_fuel_sites):
    print(f"{len(mixed_fuel_sites)} site(s) have BMUs spanning more than one fuel type - flagged, not silently summed away:")
    display(site_bmu_map[site_bmu_map["dictionary_id"].isin(mixed_fuel_sites)]
            .sort_values(["dictionary_id"])[["dictionary_id", "name", "ngc_bmu_id", "fuel_type"]])
else:
    print("No sites span more than one fuel type across their BMUs - the sum-per-site assumption holds cleanly.")

13 site(s) have BMUs spanning more than one fuel type - flagged, not silently summed away:


,dictionary_id,name,ngc_bmu_id,fuel_type
2,10001,Didcot,DIDC01G,OCGT
3,10001,Didcot,DIDC02G,OCGT
4,10001,Didcot,DIDC03G,OCGT
5,10001,Didcot,DIDC04G,OCGT
6,10001,Didcot,DIDCB5,CCGT
...,...,...,...,...
219,10141,Littlebrook D,LITTD3,COAL
220,10141,Littlebrook D,LITT2G,OCGT
217,10141,Littlebrook D,LITTD1,COAL
218,10141,Littlebrook D,LITTD2,COAL


In [9]:
SITE_GEN_DIR = INTERIM_DIR / "site_generation"
SITE_GEN_DIR.mkdir(parents=True, exist_ok=True)


def site_fuel_type(fuel_types):
    fuel_types = sorted(set(fuel_types))
    return fuel_types[0] if len(fuel_types) == 1 else "MIXED (" + "/".join(fuel_types) + ")"


def aggregate_site(dictionary_id, bmus, longitude, latitude):
    """Sum half-hourly generation across a site's BMUs. Returns None if none of
    the site's BMUs have any raw data pulled yet (e.g. pull still in progress)."""
    frames = []
    for bmu in bmus:
        path = RAW_DIR / f"{bmu}.csv"
        if path.exists():
            df = pd.read_csv(path, usecols=["settlementDate", "settlementPeriod", "quantity"])
            frames.append(df)
    if not frames:
        return None
    combined = pd.concat(frames, ignore_index=True)
    # Guard against any duplicate rows from a chunk retried after a partial write
    combined = combined.drop_duplicates()
    site_series = (
        combined.groupby(["settlementDate", "settlementPeriod"], as_index=False)["quantity"]
        .sum()
        .assign(dictionary_id=dictionary_id, longitude=longitude, latitude=latitude)
    )
    return site_series


def build_all_site_series(site_bmu_map, save=True):
    site_summaries = []
    for dictionary_id, group in site_bmu_map.groupby("dictionary_id"):
        longitude, latitude = group["longitude"].iloc[0], group["latitude"].iloc[0]
        fuel_type = site_fuel_type(group["fuel_type"])
        series = aggregate_site(dictionary_id, group["ngc_bmu_id"].tolist(), longitude, latitude)
        if series is None:
            continue
        if save:
            series.to_csv(SITE_GEN_DIR / f"{dictionary_id}.csv", index=False)
        site_summaries.append({
            "dictionary_id": dictionary_id,
            "name": group["name"].iloc[0],
            "longitude": longitude,
            "latitude": latitude,
            "fuel_type": fuel_type,
            "n_bmus": group["ngc_bmu_id"].nunique(),
            "n_half_hours": len(series),
            "earliest_settlement_date": series["settlementDate"].min(),
            "latest_settlement_date": series["settlementDate"].max(),
        })
    return pd.DataFrame(site_summaries)


site_summary = build_all_site_series(site_bmu_map)
print(f"Sites with at least some pulled data: {len(site_summary)} / {site_bmu_map['dictionary_id'].nunique()}")
site_summary.head()

Sites with at least some pulled data: 189 / 213


,dictionary_id,name,longitude,latitude,fuel_type,n_bmus,n_half_hours,earliest_settlement_date,latest_settlement_date
0,10000,Rothes Bio-Plant CHP,-3.603516,57.480403,BIOMASS,2,130808,2019-02-01,2026-07-27
1,10001,Didcot,-1.267570,51.623630,MIXED (CCGT/OCGT),6,130808,2019-02-01,2026-07-27
2,10004,Drax,-0.996631,53.736634,MIXED (BIOMASS/COAL/OCGT),9,130808,2019-02-01,2026-07-27
3,10007,Fiddlers Ferry,-2.823486,53.350551,MIXED (COAL/OCGT),6,46560,2019-02-01,2021-09-30
4,10010,Lynemouth Generator,-1.520830,55.204170,BIOMASS,3,130808,2019-02-01,2026-07-27


**Observations (full-fleet run):** 213 sites from the join, of which 189 have at least some pulled generation data — the other 24 sites' BMUs all returned zero rows across every year attempted, consistent with fully decommissioned/never-generating capacity rather than a pull failure (their manifest rows are all `success` with `rows_returned` of 0, not `failed`).

13 sites were flagged with BMUs spanning more than one fuel type (e.g. Didcot spans CCGT/OCGT, Drax spans BIOMASS/COAL/OCGT) — these are real multi-technology sites, not a data error, but summing their output blends distinct generation signals into one series labelled `MIXED (...)`. Worth deciding with Simon whether these are kept as their own "mixed" category, split back into separate series per fuel type (breaking the site-as-spatial-unit rule), or excluded — not decided here.

## Step 4 — Convex hull classification

Sites on the convex hull of the GB site layout are always kept in training (a spatial model has no way to interpolate a boundary point from surrounding sites, since there's nothing beyond it to interpolate from). Interior sites are eligible for holdout. This is a geometric classification only — no holdout proportions or thresholds are picked here.

In [10]:
from scipy.spatial import ConvexHull

coords = site_summary[["longitude", "latitude"]].to_numpy()
hull = ConvexHull(coords)
hull_site_idx = set(hull.vertices)  # indices into `coords` / `site_summary`

site_summary["hull_status"] = ["boundary" if i in hull_site_idx else "interior" for i in range(len(site_summary))]

print(site_summary["hull_status"].value_counts())
print(f"\n{len(hull_site_idx)} boundary sites (always training), {len(site_summary) - len(hull_site_idx)} interior sites (holdout-eligible)")

hull_status
interior    178
boundary     11
Name: count, dtype: int64

11 boundary sites (always training), 178 interior sites (holdout-eligible)


**Observations:** 11 boundary sites (always training), 178 interior sites (holdout-eligible), out of the 189 sites with data. The boundary is a small fraction of the fleet, which is expected for a convex hull over a roughly filled-in national footprint — most of the interesting spatial interpolation happens well inside GB's coastline, not at its extremities.

## Step 5 — Site-level fuel-type coverage

Site counts per fuel type, after multi-BMU sites have collapsed to one row each — these numbers will be smaller than the BMU-level counts from the exploration notebook, by design. **No stratification threshold is picked here** — this is a report to hand to Simon, who confirms the threshold before any holdout list is built.

In [11]:
fuel_type_coverage = (
    site_summary.groupby(["fuel_type", "hull_status"]).size().unstack(fill_value=0)
)
fuel_type_coverage["total"] = fuel_type_coverage.sum(axis=1)
fuel_type_coverage = fuel_type_coverage.sort_values("total", ascending=False)

fuel_type_coverage.to_csv(REF_DIR / "site_fuel_type_coverage.csv")
fuel_type_coverage

hull_status,boundary,interior,total
fuel_type,,,
WIND,9,102,111
CCGT,0,34,34
NPSHYD,0,13,13
NUCLEAR,1,7,8
OCGT,1,3,4
PS,0,4,4
BIOMASS,0,3,3
MIXED (CCGT/OCGT),0,3,3
MIXED (COAL/OCGT),0,3,3


**Observations:** WIND dominates at 111 sites (59% of the 189 with data) — CCGT (34) and NPSHYD (13) are the next largest, then a long tail of single-digit fuel types plus the 13 `MIXED` sites from Step 3. This is the mixed-fleet-scope tension flagged up front: WIND alone would give a strong single-technology story, but the moment other fuel types are included at these small counts, per-fuel-type stratified holdout gets thin fast for anything outside WIND/CCGT/NPSHYD.

**Data-quality note, not corrected here:** the table has both `WIND` (111 sites) and `Wind` (1 site) as separate rows — a genuine case-inconsistency in the OSUKED source data (visible already in the exploration notebook's BMU-level counts), not introduced by this pipeline. Left as-is pending Simon's confirmation of whether to normalise it before stratification.

## Step 6 — History-length distribution

Spread of available history per site, and a check of whether the ~2019-02-01 floor the 6-site exploration sample suggested actually holds across the full fleet, or was a coincidence of that small sample. **No minimum-history cutoff is applied here** — report only, pending Simon's confirmation.

In [12]:
site_summary["earliest_settlement_date"] = pd.to_datetime(site_summary["earliest_settlement_date"])
site_summary["latest_settlement_date"] = pd.to_datetime(site_summary["latest_settlement_date"])

history_stats = site_summary["earliest_settlement_date"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print("Per-site earliest available settlement date, distribution:")
print(history_stats)

print(f"\nEarliest start date across all sites: {site_summary['earliest_settlement_date'].min().date()}")
print(f"Latest start date across all sites (i.e. shortest-history site): {site_summary['earliest_settlement_date'].max().date()}")
print(f"Median start date: {site_summary['earliest_settlement_date'].median().date()}")

# How many sites start right around the exploration sample's apparent floor?
near_floor = site_summary["earliest_settlement_date"].between("2019-01-25", "2019-02-05")
print(f"\nSites starting within +/-5 days of 2019-02-01: {near_floor.sum()} / {len(site_summary)} ({near_floor.mean():.1%})")

site_summary.groupby("fuel_type")["earliest_settlement_date"].agg(["min", "median", "max", "count"])

Per-site earliest available settlement date, distribution:
count                           189
mean     2019-03-12 17:23:48.571428
min             2019-02-01 00:00:00
10%             2019-02-01 00:00:00
25%             2019-02-01 00:00:00
50%             2019-02-01 00:00:00
75%             2019-02-01 00:00:00
90%             2019-02-01 00:00:00
max             2022-02-16 00:00:00
Name: earliest_settlement_date, dtype: object

Earliest start date across all sites: 2019-02-01
Latest start date across all sites (i.e. shortest-history site): 2022-02-16
Median start date: 2019-02-01

Sites starting within +/-5 days of 2019-02-01: 179 / 189 (94.7%)


,min,median,max,count
fuel_type,,,,
BIOMASS,2019-02-01,2019-02-01,2019-02-01,3
CCGT,2019-02-01,2019-02-01,2019-02-01,34
MIXED (BIOMASS/COAL/OCGT),2019-02-01,2019-02-01,2019-02-01,1
MIXED (CCGT/COAL/OCGT),2019-02-01,2019-02-01,2019-02-01,1
MIXED (CCGT/OCGT),2019-02-01,2019-02-01,2019-02-01,3
MIXED (COAL/OCGT),2019-02-01,2019-02-01,2019-02-01,3
NPSHYD,2019-02-01,2019-02-01,2019-02-01,13
NUCLEAR,2019-02-01,2019-02-01,2019-02-01,8
OCGT,2019-02-01,2019-02-01,2019-02-01,4


In [13]:
# Caveat check: EXTRACTION_START_YEAR (2015) bounds what we could possibly observe.
# If a meaningful number of sites' earliest date sits right at that floor, it's a
# signal the true floor may be earlier still and the search window should be
# extended (the manifest makes that a cheap re-run, not a rebuild).
at_search_floor = site_summary["earliest_settlement_date"].between(f"{EXTRACTION_START_YEAR}-01-01", f"{EXTRACTION_START_YEAR}-01-07")
print(f"Sites whose earliest date is within the first week of the {EXTRACTION_START_YEAR} search floor: {at_search_floor.sum()} / {len(site_summary)}")
if at_search_floor.sum() > 0:
    print("-> non-trivial: the true history floor for these sites may predate our search window. Consider lowering EXTRACTION_START_YEAR and re-running (manifest will only fetch the newly-added earlier chunks).")
else:
    print("-> none/negligible: the search floor does not appear to be truncating real history.")

Sites whose earliest date is within the first week of the 2015 search floor: 0 / 189
-> none/negligible: the search floor does not appear to be truncating real history.


**Observations:** the ~2019-02-01 floor from the 6-site exploration sample **holds at full-fleet scale** — 179 of 189 sites (94.7%) start within five days of 2019-02-01, regardless of fuel type (confirmed across BIOMASS, CCGT, NPSHYD, NUCLEAR, OCGT, PS, RECIPROCATING and the MIXED sites, all showing min=median=2019-02-01 in the per-fuel-type breakdown above). This confirms it's a platform-wide Insights Solution backfill floor, not a per-site commissioning date or a quirk of the small exploration sample.

The exceptions are entirely within WIND: its max start date is 2022-02-16, i.e. some wind sites genuinely only have history from when they were commissioned, later than the platform floor — which is the one part of the fleet where the original "newer renewables, shorter history" intuition actually shows up.

The search-floor caveat check confirms the chosen `EXTRACTION_START_YEAR = 2015` wasn't truncating anything real: zero sites have their earliest date sitting at the 2015 boundary, so there's no evidence data exists any earlier that we failed to ask for.

## Step 7 — Parquet consolidation (derived cache, re-runnable)

Reads every per-site CSV under `data/interim/site_generation/` and writes one consolidated parquet file for the training notebook to load quickly. **The CSVs remain the source of truth** — this file is a disposable, regenerable convenience cache, safe to delete and rebuild at any time. Re-running this cell after further extraction progress just picks up whatever site CSVs exist at that point.

In [14]:
def consolidate_to_parquet(out_path=INTERIM_DIR / "site_generation_consolidated.parquet"):
    site_files = sorted(SITE_GEN_DIR.glob("*.csv"))
    if not site_files:
        print("No site CSVs found yet - nothing to consolidate.")
        return None

    frames = [pd.read_csv(f) for f in site_files]
    consolidated = pd.concat(frames, ignore_index=True)

    # Attach fuel_type and hull_status for convenience at load time in the training notebook
    consolidated = consolidated.merge(
        site_summary[["dictionary_id", "fuel_type", "hull_status", "name"]],
        on="dictionary_id", how="left",
    )
    consolidated.to_parquet(out_path, index=False)
    print(f"Wrote {len(consolidated):,} rows across {consolidated['dictionary_id'].nunique()} sites to {out_path}")
    print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")
    return consolidated


consolidated = consolidate_to_parquet()

Wrote 23,981,116 rows across 189 sites to ../data/interim/site_generation_consolidated.parquet
File size: 65.5 MB


**Observations:** ~24M rows across 189 sites consolidated into a 65.5MB parquet file — a large reduction from the raw per-BMU CSVs (43.2M rows pulled at BMU level, before the site-level sum collapses multi-BMU sites together). Comfortably small enough for the training notebook to load in full rather than needing lazy/chunked reading.

## Stopping point

This notebook deliberately stops short of three decisions, per the brief:

1. **Fuel-type stratification threshold** — Step 5's coverage table is the input; no threshold applied.
2. **Minimum-history cutoff** — Step 6's distribution is the input; no filter applied.
3. **Parquet as anything other than a derived cache** — Step 7's output is regenerable from the CSVs at any time, and should keep being treated that way.

Also acknowledged, not glossed over: the mixed-fuel-type fleet scope (Step 3's flag for any BMU-level mismatch, plus the general point that a single-fuel-type dataset would tell a cleaner geographic story) needs to be stated plainly in the eventual write-up, not just here.